# 14 — Classical Text Classification

**Learning objective.** Build a leakage-safe TF-IDF + logistic regression classifier using a scikit-learn Pipeline.

This notebook is intentionally **offline-reproducible**: the examples use local data or deterministic toy corpora so the rendered GitHub output can be trusted without hidden API calls. The focus is always **concept → inspectable representation → library implementation → result → failure modes → production implication**.

## Mental model

**labeled text → vectorizer → decision boundary → class prediction**

Focus on the transformation of information from left to right. Ask what representation changes before asking which library call implements it.

## Change one thing at a time

| Change | Immediate effect | Downstream consequence |
|---|---|---|
| Change **representation** | input geometry changes | the same classifier can behave very differently |
| Increase logistic-regression **C** | regularization weakens | weights can grow; fit rises; overfitting risk rises |
| Move vectorizer outside Pipeline before split | train/test information can leak | validation becomes optimistically biased |

> Before changing a parameter, state the expected direction of the downstream effect.

## Think before running the next cell

1. Can a stronger classifier recover information that the representation never encoded?
2. If validation improves but test drops after aggressive tuning, what happened?

### When to use
Use classical pipelines as transparent, fast baselines and often production models for short domain text.

### When not to use / caution
Do not jump to complex models before measuring a strong sparse baseline.

### Debugging lens
Separate errors into data, representation and decision-boundary problems before changing algorithms.

In [1]:
from pathlib import Path
import re, math, json, random, statistics
import numpy as np
import pandas as pd
np.random.seed(42)
random.seed(42)
pd.set_option('display.max_colwidth', 120)
DATA = Path('data')
print('Reproducibility seed: 42')

Reproducibility seed: 42


In [2]:
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix
_df=pd.read_csv(DATA/'sentiment_reviews.csv')
Xtr,Xte,ytr,yte=train_test_split(_df.text,_df.label,test_size=0.33,random_state=42,stratify=_df.label)
clf=Pipeline([
 ('tfidf',TfidfVectorizer(ngram_range=(1,2),min_df=1)),
 ('model',LogisticRegression(max_iter=1000,random_state=42))])
clf.fit(Xtr,ytr)
pred=clf.predict(Xte)
print(classification_report(yte,pred,zero_division=0))
print(confusion_matrix(yte,pred,labels=clf.classes_))

              precision    recall  f1-score   support

    negative       0.20      1.00      0.33         1
     neutral       0.00      0.00      0.00         2
    positive       0.00      0.00      0.00         2

    accuracy                           0.20         5
   macro avg       0.07      0.33      0.11         5
weighted avg       0.04      0.20      0.07         5

[[1 0 0]
 [2 0 0]
 [2 0 0]]


In [3]:
examples=['battery is fantastic','the app crashes constantly','device has a six inch screen']
print(pd.DataFrame({'text':examples,'prediction':clf.predict(examples)}).to_string(index=False))

                        text prediction
        battery is fantastic   negative
  the app crashes constantly   negative
device has a six inch screen    neutral


---
## Production takeaways
- Preserve preprocessing as part of the model contract; training/inference skew is an NLP failure mode, not an implementation detail.
- Inspect intermediate representations rather than treating tokenizers/vectorizers/models as black boxes.
- Prefer the simplest representation/model that meets quality, latency, governance, and maintenance requirements.

### What you should now be able to explain
- Use Pipeline to prevent preprocessing leakage
- Evaluate per class rather than only accuracy